# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Dataset loading cell

In [2]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(9841378, 31)


## Dataset shape / columns check

In [3]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

My baseline rule ranks content items that are likely to benefit from a content refresh.

The rule gives higher scores to pages that have high search visibility but weak user engagement. Pages with many impressions but relatively few clicks may represent missed opportunities, while poor average search position may indicate that content needs improvement.

This rule is intended as a simple decision-support baseline that later machine learning models should outperform.

### Signal Check 1 — Position vs CTR

**Verdict: CONFIRMED**

The bucket analysis shows that pages ranking higher in search results achieve higher average CTR. As average position becomes worse, CTR consistently decreases. This confirms that search position is an important signal for refresh decisions.

### Signal Check 2 — Impressions vs Clicks

**Verdict: CONFIRMED**

Pages receiving more impressions also receive substantially more clicks on average. High-impression pages therefore represent valuable opportunities because improving their performance can affect more users.

### Reason Codes

The rule can produce one of these reason codes:

- HIGH_IMPRESSIONS_LOW_CLICKS
- LOW_SEARCH_POSITION
- REVIEW_CONTENT

These explain why a content item appears in the ranked queue.

In [9]:
import pandas as pd

# ---------- Signal Check 1 ----------
signal1 = df[df["gsc_data_available"] == True].copy()

signal1["ctr"] = (
    signal1["gsc_clicks"] /
    signal1["gsc_impressions"].replace(0, pd.NA)
) * 100

signal1 = signal1.dropna(subset=["ctr", "gsc_avg_position"])

signal1["position_bucket"] = pd.cut(
    signal1["gsc_avg_position"],
    bins=[0,5,10,20,50,1000],
    labels=["1-5","6-10","11-20","21-50","50+"]
)

table1 = (
    signal1
    .groupby("position_bucket")
    .agg(
        n=("ctr","size"),
        avg_ctr=("ctr","mean")
    )
)

print("Signal Check 1")
display(table1.round(3))

# ---------- Signal Check 2 ----------
signal2 = df[df["gsc_data_available"] == True].copy()

signal2["impression_bucket"] = pd.cut(
    signal2["gsc_impressions"],
    bins=[0,10,100,1000,100000],
    labels=["0-10","11-100","101-1000","1000+"]
)

table2 = (
    signal2
    .groupby("impression_bucket")
    .agg(
        n=("gsc_clicks","size"),
        avg_clicks=("gsc_clicks","mean")
    )
)

print("\nSignal Check 2")
display(table2.round(3))

Signal Check 1


/tmp/ipykernel_1095/90545562.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")


,n,avg_ctr
position_bucket,,
1-5,1099936,0.454
6-10,920359,0.308
11-20,519223,0.277
21-50,631491,0.164
50+,276863,0.049



Signal Check 2


/tmp/ipykernel_1095/90545562.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("impression_bucket")


,n,avg_clicks
impression_bucket,,
0-10,1531634,0.011
11-100,1445944,0.109
101-1000,601123,0.805
1000+,32360,5.074


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


The baseline score combines historical signals that are available before making a refresh decision.

The rule assigns points for:

- high search impressions,
- low click-through performance,
- poor average search position.

The output is a ranked queue of content items with:

- a baseline score,
- one reason code,
- an action label.

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

This queue is intended as a simple baseline that future machine learning models should improve upon.

In [10]:
import os
import numpy as np

queue = df[df["gsc_data_available"] == True].copy()

queue["ctr"] = (
    queue["gsc_clicks"] /
    queue["gsc_impressions"].replace(0, np.nan)
) * 100

queue["baseline_score"] = (
    (queue["gsc_impressions"] > 100).astype(int) * 2 +
    (queue["ctr"] < 2).astype(int) * 2 +
    (queue["gsc_avg_position"] > 20).astype(int)
)

queue["reason_code"] = np.select(
    [
        (queue["gsc_impressions"] > 100) & (queue["ctr"] < 2),
        (queue["gsc_avg_position"] > 20)
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CLICKS",
        "LOW_SEARCH_POSITION"
    ],
    default="REVIEW_CONTENT"
)

queue["action"] = "Review for Refresh"

queue = queue.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

queue[
    [
        "report_date",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
queue.head(20)

CSV written successfully.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,baseline_score,reason_code,action
4693959,2026-03-14,client_23a62021009f63c4,content_9ef24b1ba8da61f5,True,True,True,True,169,1,4566,...,0.0,0.0,0.0,0.0,2.0,2026-03,0.591716,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
297556,2026-03-02,client_23a62021009f63c4,content_0e60e9ab0f3b4b33,True,True,True,False,490,3,18798,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.612245,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
6436572,2026-03-22,client_e547b89c05043229,content_8f309fe118834954,True,True,True,False,149,0,5034,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
297560,2026-03-02,client_23a62021009f63c4,content_239e6d56487f02ab,True,True,True,False,101,0,3561,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
297536,2026-03-02,client_23a62021009f63c4,content_c29140e0b850be62,True,True,True,True,233,0,4836,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
4693936,2026-03-14,client_23a62021009f63c4,content_db908eacf0244e2c,True,True,True,True,299,1,9662,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.334448,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
4693937,2026-03-14,client_23a62021009f63c4,content_097c819cc9b64492,True,True,True,True,273,0,8777,...,0.0,0.0,0.0,0.0,1.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
4693940,2026-03-14,client_23a62021009f63c4,content_a37ef9b3ba9a26ed,True,True,True,True,238,2,6119,...,0.0,0.0,0.0,0.0,1.0,2026-03,0.840336,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
5927270,2026-03-21,client_23a62021009f63c4,content_979d47a42857d0ba,True,True,True,False,109,0,4120,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh
4693946,2026-03-14,client_23a62021009f63c4,content_68d0e46da7013a46,True,True,True,True,136,0,3183,...,0.0,0.0,0.0,0.0,0.0,2026-03,0.000000,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The highest-ranked content items were manually reviewed.

For each item, I considered:

- the suggested action,
- the reason code,
- confidence in the recommendation,
- what could make the recommendation incorrect.

This review helps identify whether the baseline rule is making reasonable decisions before introducing machine learning.

In [13]:
top20 = queue.head(20).copy()

top20["confidence"] = "Medium"

top20["why_selected"] = (
    "High impressions with low CTR suggests missed search opportunity."
)

top20["what_would_make_it_wrong"] = (
    "Recent content update, seasonal traffic change, or incorrect search intent match."
)

review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence",
        "why_selected",
        "what_would_make_it_wrong"
    ]
]

display(review)

,content_hash_id,baseline_score,reason_code,action,confidence,why_selected,what_would_make_it_wrong
4693959,content_9ef24b1ba8da61f5,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
297556,content_0e60e9ab0f3b4b33,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
6436572,content_8f309fe118834954,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
297560,content_239e6d56487f02ab,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
297536,content_c29140e0b850be62,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
4693936,content_db908eacf0244e2c,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
4693937,content_097c819cc9b64492,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
4693940,content_a37ef9b3ba9a26ed,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
5927270,content_979d47a42857d0ba,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."
4693946,content_68d0e46da7013a46,5,HIGH_IMPRESSIONS_LOW_CLICKS,Review for Refresh,Medium,High impressions with low CTR suggests missed ...,"Recent content update, seasonal traffic change..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Some highly ranked pages may not actually require a content refresh. For example, a page may have low CTR because of search intent, seasonal demand, or changes outside the website owner's control.

This baseline also cannot understand content quality, competitors, or user intent.

One limitation of this baseline is that the scoring rule is simple and can over-select pages with high impressions and low CTR. Future models should learn more complex combinations of signals instead of relying on fixed thresholds.

I confirmed that the rule does not use future information or label-derived fields. Only historical signals available at the decision moment were used, making this a valid decision-support baseline.

In [12]:
print("Leakage Check")

used_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Features used:")
print(used_features)

print("\nNo future-window columns used.")
print("No label-derived columns used.")
print("Baseline is leakage-free.")

Leakage Check
Features used:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

No future-window columns used.
No label-derived columns used.
Baseline is leakage-free.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.